# Huấn luyện mô hình phân loại câu hỏi – Student FAQ Router

Notebook này thực hiện **Step 1 (Data preparation)** và **Step 2 (AI modeling)** của dự án. Mỗi mục ứng với một gạch đầu dòng trong đề của giảng viên, kết quả (bảng, biểu đồ) dùng trực tiếp cho báo cáo.

**Cách chạy:** chỉnh các tham số ở mục 0 rồi chọn *Kernel → Restart & Run All*.

**Kết quả:**
- `models/<VERSION>/model.joblib` và `metadata.json`: model và ngưỡng mà ứng dụng dùng.
- `reports/<VERSION>/`: biểu đồ và bảng cho báo cáo.

**Giai đoạn** (`config.TRA_LOI_TU_DONG`):
- `False` – **giai đoạn 1**: mô hình phân loại câu hỏi và chuyển về đúng phòng ban. Mục 2.7 (kho FAQ) được bỏ qua.
- `True` – **giai đoạn 2**: thêm trả lời tự động bằng kho FAQ; notebook chọn thêm ngưỡng tương đồng `s_min`.

**Chế độ demo:** đặt biến môi trường `FAQ_ROUTER_DEMO=1` để chạy với dữ liệu mẫu `data/sample/`. Kết quả demo chỉ để kiểm tra code, **không dùng cho báo cáo**.

## 0. Cấu hình

In [ ]:
import os, sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
warnings.filterwarnings("ignore", category=UserWarning)

import config
DEMO = config.DEMO

# ====================== THAM SỐ – nhóm chỉnh ở đây ======================
VERSION = "demo" if DEMO else "v1"     # phiên bản model sẽ tạo: v1, v2, ...
PREV_VERSION = None                    # phiên bản cũ để so sánh, ví dụ "v1" khi huấn luyện v2
DATA_PATH = config.DATA_DIR / ("sample/questions_sample.csv" if DEMO else "questions.csv")
LOG_PATH = None if DEMO else config.DATA_DIR / "log_labeled.csv"   # câu hỏi từ nhật ký đã gán nhãn (Step 4)
KAPPA_PATH = config.DATA_DIR / ("sample/kappa_sample.csv" if DEMO else "kappa_check.csv")
FAQ_PATH = config.FAQ_PATH
SPLIT_DIR = config.DATA_DIR / ("sample/splits" if DEMO else "splits")   # nơi lưu id tập test đã đóng băng
REPORT_DIR = config.REPORT_ROOT / VERSION
MUC_TIEU_PRECISION = 0.90              # precision tối thiểu của câu trả lời tự động khi chọn ngưỡng
SEED = 42
# =========================================================================

REPORT_DIR.mkdir(parents=True, exist_ok=True)
GIAI_DOAN = 2 if config.TRA_LOI_TU_DONG else 1
THEO_PHONG_BAN = GIAI_DOAN == 1   # giai đoạn 1: "đúng" nghĩa là chuyển đúng phòng ban
print("Phiên bản:", VERSION, "| Giai đoạn:", GIAI_DOAN, "| Dữ liệu:", DATA_PATH, "| Demo:", DEMO)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

import training as T
from preprocess import khong_dau
from router import doc_faq

pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.dpi"] = 110
TEN_NHAN = {k: v[0] for k, v in config.NHAN.items()}

def luu_hinh(ten):
    plt.tight_layout()
    plt.savefig(REPORT_DIR / ten, dpi=150, bbox_inches="tight")
    plt.show()

## 1. Step 1 – Data preparation

### 1.1. Data requirements & Data collection
Đọc dữ liệu và kiểm tra cấu trúc: các cột bắt buộc `id, text, label, source`; nhãn phải thuộc các nhãn trong `config.NHAN`; `id` không trùng. Bảng dưới cho biết số câu theo **nguồn** và **nhãn**, dùng cho mục *Data collection* của báo cáo.

In [ ]:
df_goc = T.doc_du_lieu(DATA_PATH)
if (df_goc.source == "demo").any():
    print("⚠ Dữ liệu đang chứa câu MẪU (source = demo) — KHÔNG dùng kết quả này cho báo cáo.")
print("Tổng số câu:", len(df_goc))
pd.crosstab(df_goc.label, df_goc.source, margins=True, margins_name="Tổng")

### 1.2. Data labeling – độ đồng thuận giữa hai người gán nhãn
Hai thành viên gán nhãn độc lập cùng khoảng 100 câu (file có cột `text, label_a, label_b`). Hệ số **Cohen's Kappa** ≥ 0.8 là đồng thuận tốt. Các câu bất đồng được thảo luận để sửa hướng dẫn gán nhãn.

In [ ]:
if Path(KAPPA_PATH).exists():
    kappa, bat_dong = T.tinh_kappa(KAPPA_PATH)
    print(f"Cohen's Kappa = {kappa:.3f} trên {pd.read_csv(KAPPA_PATH).shape[0]} câu; {len(bat_dong)} câu bất đồng:")
    display(bat_dong)
else:
    print(f"Chưa có {KAPPA_PATH}. Tạo theo mẫu data/templates/kappa_template.csv.")

### 1.3. Data cleaning
Chuẩn hóa Unicode (NFC), bỏ khoảng trắng thừa, bỏ câu rỗng/chưa gán nhãn, bỏ câu trùng sau khi chuẩn hóa, bỏ câu quá ngắn (< 3 từ, trừ lớp KHAC). Câu **trùng nhau nhưng khác nhãn** được liệt kê để nhóm rà lại.

In [ ]:
df, bang_sach, xung_dot = T.lam_sach(df_goc)
print(f"Trước làm sạch: {len(df_goc)} câu → sau làm sạch: {len(df)} câu")
display(bang_sach)
if len(xung_dot):
    print("Câu trùng nhưng khác nhãn — cần rà lại:")
    display(xung_dot)

### 1.4. Phân tích khám phá dữ liệu (EDA)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.label.map(TEN_NHAN).value_counts().sort_values().plot.barh(ax=ax[0], color="#4C78A8")
ax[0].set_title("Số câu mỗi chủ đề"); ax[0].set_xlabel("Số câu")
df.text.str.split().str.len().plot.hist(ax=ax[1], bins=20, color="#F58518")
ax[1].set_title("Độ dài câu hỏi"); ax[1].set_xlabel("Số từ"); ax[1].set_ylabel("Số câu")
luu_hinh("eda.png")
print(f"Tỉ lệ câu không dấu: {df.text.map(khong_dau).mean():.1%}")
print(f"Độ dài trung bình: {df.text.str.split().str.len().mean():.1f} từ")

### 1.5. Dataset splitting
- Tập **test** chiếm 20%, chỉ gồm **câu hỏi thật** (nguồn trong `config.NGUON_TEST`), giữ tỉ lệ các nhãn.
- Danh sách id của tập test được **đóng băng** ở `SPLIT_DIR/test_ids.txt`: các lần huấn luyện sau (v2, v3…) dùng lại đúng tập test này để so sánh công bằng.
- Câu do nhóm tự viết (`team`) và câu lấy từ kho FAQ (`faq`) chỉ nằm trong tập train.
- Nếu đã có câu hỏi từ nhật ký được gán nhãn (`LOG_PATH`, Step 4): một nửa bổ sung vào train, một nửa làm **tập test thực tế**.
- Trên tập train dùng kiểm định chéo 5-fold để chọn mô hình, tham số và ngưỡng, nên không cần tập validation riêng.

In [ ]:
train, test, thong_bao = T.chia_tap(df, SPLIT_DIR, seed=SEED)
print(thong_bao)

test_thuc_te = None
if LOG_PATH is not None and Path(LOG_PATH).exists():
    df_log, _, _ = T.lam_sach(T.doc_du_lieu(LOG_PATH))
    log_train, test_thuc_te = T.chia_nhat_ky(df_log, SPLIT_DIR, seed=SEED)
    train = pd.concat([train, log_train], ignore_index=True)
    print(f"Nhật ký: {len(log_train)} câu bổ sung vào train, {len(test_thuc_te)} câu làm tập test thực tế.")

bang_chia = pd.DataFrame({"Train": train.label.value_counts(), "Test": test.label.value_counts()}).fillna(0).astype(int)
if test_thuc_te is not None:
    bang_chia["Test thực tế"] = test_thuc_te.label.value_counts()
bang_chia.loc["Tổng"] = bang_chia.sum()
bang_chia

### 1.6. Data augmentation & Data pre-processing
- **Tiền xử lý** (`preprocess.tien_xu_ly`, dùng chung với ứng dụng): chuẩn hóa → ẩn MSSV/SĐT/email → bỏ link, dấu câu, emoji → mở rộng teencode → tách từ tiếng Việt (underthesea). **Không loại stopword** vì các từ như “khi nào”, “bao nhiêu” mang thông tin về chủ đề.
- **Tăng cường**: thêm bản **không dấu** của các câu có dấu, chỉ cho dữ liệu huấn luyện. Khi kiểm định chéo, bản không dấu chỉ được thêm vào phần train của mỗi fold, nên không bị rò rỉ sang phần đánh giá.

In [ ]:
train = T.them_cot_xu_ly(train)
test = T.them_cot_xu_ly(test)
if test_thuc_te is not None:
    test_thuc_te = T.them_cot_xu_ly(test_thuc_te)
X_tmp, _ = T.tap_huan_luyen(train)
print(f"Train: {len(train)} câu gốc + {len(X_tmp) - len(train)} bản không dấu = {len(X_tmp)} câu để huấn luyện")
train[["text", "x", "x_bodau"]].head(6).rename(columns={
    "text": "Câu gốc", "x": "Sau tiền xử lý", "x_bodau": "Bản không dấu (tăng cường)"})

## 2. Step 2 – AI modeling

### 2.1. Model selection
So sánh 3 mô hình trên cùng đặc trưng TF-IDF (tham số mặc định), bằng kiểm định chéo 5-fold, thước đo chính **Macro-F1** (mỗi chủ đề quan trọng như nhau):
- **Naive Bayes** – mô hình cơ sở cho phân loại văn bản;
- **Logistic Regression** – mô hình chính, cho ra xác suất dùng làm độ tin cậy;
- **Linear SVM** – tìm đường biên có lề rộng nhất giữa các chủ đề.

Quy tắc chọn: mô hình có Macro-F1 cao nhất; nếu Logistic Regression kém mô hình tốt nhất không quá 0.02 thì vẫn chọn Logistic Regression vì hệ thống cần xác suất.

In [ ]:
so_sanh = T.so_sanh_mo_hinh(train, seed=SEED)
so_sanh.round(3).to_csv(REPORT_DIR / "so_sanh_mo_hinh.csv", index=False)
display(so_sanh.round(3))

ax = so_sanh.set_index("Mô hình")[["Macro-F1 (CV)", "Macro-F1 câu không dấu"]].plot.bar(
    rot=0, figsize=(8, 4), color=["#4C78A8", "#F58518"], yerr={"Macro-F1 (CV)": so_sanh["Độ lệch chuẩn"].values})
ax.set_ylim(0, 1); ax.set_title("So sánh mô hình (kiểm định chéo 5-fold)")
luu_hinh("so_sanh_mo_hinh.png")

TEN_MO_HINH, LY_DO = T.chon_mo_hinh(so_sanh)
print("Mô hình được chọn:", TEN_MO_HINH, "—", LY_DO)

### 2.2. Thí nghiệm nhỏ (ablation)
Mỗi lần bỏ một yếu tố của bước tiền xử lý để xem yếu tố đó đóng góp bao nhiêu. Cột *câu không dấu* cho biết mô hình có chịu được cách gõ không dấu của sinh viên hay không.

In [ ]:
ablation = T.thi_nghiem_nho(train, TEN_MO_HINH)
ablation.round(3).to_csv(REPORT_DIR / "thi_nghiem_nho.csv", index=False)
ablation.round(3)

### 2.3. Model tuning
Tìm kiếm lưới (grid search) với kiểm định chéo 5-fold trên: tham số điều chuẩn của mô hình (`C` hoặc `alpha`), đặc trưng theo **từ** (n-gram 1–2) hay theo **ký tự** (n-gram 2–5), `min_df`, `sublinear_tf`.

In [ ]:
THAM_SO, luoi = T.tim_tham_so(train, TEN_MO_HINH, seed=SEED)
luoi.to_csv(REPORT_DIR / "tim_tham_so.csv", index=False)
print(f"Đã thử {len(luoi)} tổ hợp. Tham số tốt nhất (Macro-F1 CV = {luoi.loc[0, 'Macro-F1 (CV)']:.3f}):")
print(THAM_SO)
luoi.head(10).round(3)

### 2.4. Chọn ngưỡng tin cậy τ
Câu có độ tin cậy ≥ τ được **xử lý tự động**: ở giai đoạn 1 là chuyển thẳng đến phòng ban phụ trách, ở giai đoạn 2 là trả lời bằng kho FAQ. Câu có độ tin cậy < τ được chuyển cho **Phòng Công tác sinh viên phân loại thủ công** (đúng bước 2 của Quy trình QT_11_CTSV).

Dùng xác suất **ngoài mẫu** từ kiểm định chéo (mỗi câu được dự đoán bởi mô hình chưa thấy câu đó). Với mỗi τ: *coverage* là tỉ lệ câu được xử lý tự động, *precision* là tỉ lệ đúng trong các câu đó (giai đoạn 1 tính **theo phòng ban**). Chọn τ nhỏ nhất đạt precision ≥ `MUC_TIEU_PRECISION`.

In [ ]:
cv_cuoi = T.danh_gia_cheo(train, TEN_MO_HINH, THAM_SO, xac_suat=True, seed=SEED)
TAU, bang_tau, ghi_chu_tau = T.chon_nguong(cv_cuoi["oof"], cv_cuoi["lop"], train.label, MUC_TIEU_PRECISION,
                                           theo_phong_ban=THEO_PHONG_BAN)
print(ghi_chu_tau)
bang_tau.round(3).to_csv(REPORT_DIR / "chon_nguong.csv", index=False)

ax = bang_tau.plot(x="tau", y=["coverage", "precision"], marker="o", figsize=(8, 4), color=["#4C78A8", "#54A24B"])
ax.axvline(TAU, ls="--", color="grey"); ax.axhline(MUC_TIEU_PRECISION, ls=":", color="#E45756")
ax.set_ylim(0, 1.05); ax.set_xlabel("Ngưỡng τ"); ax.set_title("Coverage và precision của câu được xử lý tự động theo τ")
luu_hinh("chon_nguong.png")

### 2.5. Huấn luyện mô hình cuối cùng trên toàn bộ tập train

In [ ]:
model = T.huan_luyen_cuoi(train, TEN_MO_HINH, THAM_SO)
print(model)

### 2.6. Model evaluation – đánh giá trên tập test (một lần duy nhất)
Các chỉ số: Accuracy và Macro-F1 theo chủ đề, **Accuracy theo phòng ban**, recall lớp KHAC, tỉ lệ và precision của câu được xử lý tự động tại τ, Macro-F1 khi câu test bị bỏ dấu (độ bền), thời gian dự đoán.

In [ ]:
ev = T.danh_gia_tap(model, test, TAU, "Test", theo_phong_ban=THEO_PHONG_BAN)
bang_test = [ev["chi_so"]]
if test_thuc_te is not None:
    bang_test.append(T.danh_gia_tap(model, test_thuc_te, TAU, "Test thực tế", theo_phong_ban=THEO_PHONG_BAN)["chi_so"])
bang_test = pd.DataFrame(bang_test)
THOI_GIAN_MS = T.do_thoi_gian(model, test.text)
bang_test.round(3).to_csv(REPORT_DIR / "danh_gia_test.csv", index=False)
display(bang_test.round(3))
print(f"Thời gian tiền xử lý + dự đoán trung bình: {THOI_GIAN_MS:.1f} ms/câu")
print(ev["bao_cao"])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
nhan = [n for n in config.NHAN if n in set(test.label) | set(ev["du_doan"])]
ConfusionMatrixDisplay.from_predictions(test.label, ev["du_doan"], labels=nhan, ax=ax,
                                        xticks_rotation=45, cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix trên tập test"); ax.set_xlabel("Dự đoán"); ax.set_ylabel("Nhãn đúng")
luu_hinh("confusion_matrix.png")

**Kết quả chuyển phòng ban trên tập test** — mỗi hàng là phòng ban đúng, mỗi cột là nơi hệ thống chuyển câu hỏi đến. Câu có độ tin cậy < τ được tính vào cột *Phòng Công tác sinh viên (phân loại thủ công)*.

In [ ]:
proba_test = model.predict_proba(test.x)
chuyen_den = np.where((proba_test.max(1) < TAU), f"{config.PHONG_BAN_XEM_LAI} (phân loại thủ công)",
                      [config.phong_ban(n) for n in ev["du_doan"]])
bang_pb = pd.crosstab(test.label.map(config.phong_ban).rename("Phòng ban đúng"),
                      pd.Series(chuyen_den, name="Hệ thống chuyển đến"), margins=True, margins_name="Tổng")
bang_pb.to_csv(REPORT_DIR / "chuyen_phong_ban.csv")
bang_pb

**Phân tích lỗi:** các câu dự đoán sai (lưu ở `reports/<VERSION>/loi_du_doan.csv`). Nhóm điền thêm cột nguyên nhân (câu nhiều ý, nhãn mơ hồ, từ hiếm, lỗi chính tả…) cho báo cáo.

In [ ]:
ev["loi"].to_csv(REPORT_DIR / "loi_du_doan.csv", index=False)
ev["loi"].round(3)

**Giải thích mô hình:** các từ/cụm từ có trọng số cao nhất của mỗi chủ đề.

In [ ]:
tu = T.tu_dac_trung(model, 10)
tu.to_csv(REPORT_DIR / "tu_dac_trung.csv", index=False)
tu

### 2.7. (Giai đoạn 2) Đánh giá bước chọn câu FAQ và ngưỡng s_min
Chỉ chạy khi `config.TRA_LOI_TU_DONG = True`. Trên các câu có ghi `faq_id` (câu FAQ đúng), hệ thống dự đoán chủ đề rồi tìm câu FAQ giống nhất trong chủ đề đó. Chọn `s_min` nhỏ nhất đạt precision ≥ `MUC_TIEU_PRECISION`; câu không đủ giống FAQ nào sẽ được chuyển đến phòng ban.

In [ ]:
S_MIN = None
if GIAI_DOAN == 1:
    print("Giai đoạn 1 (chỉ phân loại và chuyển phòng ban): bỏ qua bước chọn câu FAQ.")
else:
    faq = doc_faq(FAQ_PATH)
    print(f"Kho FAQ: {len(faq)} câu")
    display(faq.label.map(TEN_NHAN).value_counts().rename("Số câu FAQ"))
    S_MIN, bang_s, ghi_chu_s = T.danh_gia_faq(model, faq, pd.concat([train, test]), MUC_TIEU_PRECISION)
    print(ghi_chu_s)
    if bang_s is None:
        S_MIN = 0.2
        print("Tạm dùng s_min = 0.2. Ghi faq_id cho khoảng 50 câu để chọn ngưỡng theo dữ liệu.")
    else:
        bang_s.round(3).to_csv(REPORT_DIR / "chon_nguong_faq.csv", index=False)
        display(bang_s.round(3))

### 2.8. So sánh với phiên bản trước (khi huấn luyện lại – Step 4)
Đặt `PREV_VERSION = "v1"` ở mục 0 khi huấn luyện v2. Hai phiên bản được đánh giá trên **cùng** tập test gốc (đã đóng băng) và tập test thực tế. Chỉ thay model trong `config.py` nếu phiên bản mới không kém hơn.

In [ ]:
if PREV_VERSION:
    import json, joblib
    thu_muc_cu = config.MODEL_ROOT / PREV_VERSION
    model_cu = joblib.load(thu_muc_cu / "model.joblib")
    tau_cu = json.loads((thu_muc_cu / "metadata.json").read_text(encoding="utf-8"))["nguong_tin_cay"]
    hang = []
    for ten_tap, tap in [("Test gốc", test), ("Test thực tế", test_thuc_te)]:
        if tap is None:
            continue
        for pb, m, t in [(PREV_VERSION, model_cu, tau_cu), (VERSION, model, TAU)]:
            chi_so = T.danh_gia_tap(m, tap, t, theo_phong_ban=THEO_PHONG_BAN)["chi_so"]
            hang.append({"Tập": ten_tap, "Phiên bản": pb, **{k: v for k, v in chi_so.items() if k != "Tập"}})
    so_sanh_pb = pd.DataFrame(hang)
    so_sanh_pb.round(3).to_csv(REPORT_DIR / "so_sanh_phien_ban.csv", index=False)
    display(so_sanh_pb.round(3))
else:
    print("PREV_VERSION = None: bỏ qua so sánh phiên bản.")

### 2.9. Lưu model và metadata

In [ ]:
ct = ev["chi_so"]
thu_muc = T.luu_model(model, config.MODEL_ROOT / VERSION, {
    "phien_ban": VERSION, "giai_doan": GIAI_DOAN, "demo": DEMO, "du_lieu": str(Path(DATA_PATH).relative_to(ROOT)),
    "mo_hinh": TEN_MO_HINH, "ly_do_chon": LY_DO, "tham_so": THAM_SO,
    "nguong_tin_cay": TAU, "nguong_tuong_dong": S_MIN,
    "so_cau_train": len(train), "so_cau_test": len(test),
    "macro_f1_cv": float(luoi.loc[0, "Macro-F1 (CV)"]),
    "chi_so_test": {k: (None if pd.isna(v) else float(v)) for k, v in ct.items() if k not in ("Tập", "Số câu")},
    "thoi_gian_ms_moi_cau": THOI_GIAN_MS,
})
print("Đã lưu:", thu_muc)
print(f"Mô hình: {TEN_MO_HINH} | τ = {TAU:.2f} | s_min = {S_MIN} | Macro-F1 test = {ct['Macro-F1']:.3f} "
      f"| Accuracy theo phòng ban = {ct['Accuracy theo phòng ban']:.3f}")
if VERSION != config.MODEL_VERSION:
    print(f"Lưu ý: ứng dụng đang dùng MODEL_VERSION = {config.MODEL_VERSION!r} (config.py).")